In [ ]:
import sys
!{sys.executable} -m pip install ultralytics wandb

In [ ]:
import torch, yaml                                                                                                                           
from pathlib import Path                                                                                                                     
                                                                                                                                               
print("CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0))                                                        
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
                                                                                                                                               
cfg_path = Path("/workspace/merged/config/dataset.yaml")                                                                                     
cfg = yaml.safe_load(cfg_path.read_text())                                                                                                   
print("\ndataset.yaml:", cfg)                                                                                                                
                                                                                                                                               
for split in ("train", "val", "test"):  
    n = len(list((Path(cfg["path"]) / "images" / split).glob("*.jpg")))                                                                      
    print(f"{split}: {n} images")

In [ ]:
from ultralytics import YOLO                
                                                                                                                                               
model = YOLO("yolo11n.pt")
print(model.info()) 

In [ ]:
results = model.train(                      
      data="/workspace/merged/config/dataset.yaml",
      epochs=100,                                  
      imgsz=640,                                                                                                                               
      batch=64,                # V100 32GB handles this easily at imgsz=640
      device=0,                                                                                                                                
      workers=6,               # matches your 6 CPU cores
      project="/workspace/runs/detect",                  
      name="person_v11n",              
      exist_ok=False,                                                                                                                          
      save=True,                          
      save_period=10,          # checkpoint every 10 epochs                                                                                    
      patience=20,             # early stop if val doesn't improve for 20 epochs                                                               
      amp=True,                # mixed precision — ~2x speedup on V100          
      cos_lr=True,             # cosine LR schedule                                                                                            
      close_mosaic=10,         # disable mosaic last 10 epochs for clean convergence
      mosaic=1.0,                                                                                                                              
      hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,      
      fliplr=0.5, flipud=0.0,                                                                                                                  
      degrees=0.0, translate=0.1, scale=0.5,                                                                                                   
      plots=True,                                                                                                                              
      verbose=True,                                                                                                                            
  )           

In [ ]:
best = YOLO("/workspace/runs/detect/person_v11n/weights/best.pt")
metrics = best.val(                                                                                                                          
      data="/workspace/merged/config/dataset.yaml",
      split="test",                                                                                                                            
      imgsz=640,                          
      batch=64,                                                                                                                                
      device=0,                                                                                                                                
      plots=True,
  )                                                                                                                                            
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")  
print(f"Precision:  {metrics.box.mp:.4f}")                                                                                                   
print(f"Recall:     {metrics.box.mr:.4f}")

In [ ]:
best.export(format="onnx", imgsz=640, simplify=True, opset=12, dynamic=False)
  # Optional: TensorRT engine (only works if targeting same GPU arch)                                                                          
  # best.export(format="engine", imgsz=640, half=True, device=0) 

In [ ]:
import shutil                                                                                                                                
shutil.make_archive("/workspace/person_v11n_results", "tar",                                                                                 
                      "/workspace/runs/detect", "person_v11n")                                                                                 
print("Archive:", "/workspace/person_v11n_results.tar")